**withColumn, withColumnRenamed & Dropping Columns**

**withColumn:Add or Replace a Column**

withColumn() takes two arguments — the new column name and the expression. If the column name already exists, it replaces that column. If the name is new, it adds a new column. The original DataFrame is never modified.

syntax:df.withColumn('column_name',expression)


**Performance Note: withColumn() vs select()**

PySpark DataFrames are immutable, so every withColumn() call returns a new DataFrame instead of modifying the existing one.

Spark uses lazy evaluation, so multiple withColumn() calls are not executed immediately; they are added to the logical execution plan.

Chaining a few withColumn() calls (2–4) is perfectly fine and has minimal impact.

Adding many columns (5 or more) using repeated withColumn() calls can make the logical plan more complex, increasing query planning overhead.

For multiple column transformations, prefer a single select() with all expressions, as it creates a simpler logical plan and gives Spark's Catalyst Optimizer a better opportunity to optimise the query.

Use withColumn() for readability when adding one or two columns, and use select() for bulk column creation in production ETL pipelines.


**withColumnRenamed():Rename a Column**

withColumnRenamed() takes two arguments — the existing column name and the new name. It returns a new DataFrame with that column renamed. All other columns remain unchanged.

**Drop():Remove Columns**

drop() removes one or more columns from a DataFrame. It is the complement of select() — use it when you want all columns except a few.


Note:

Dropping a column that does not exist does NOT throw an error in PySpark — it just returns the DataFrame unchanged. This is different from most SQL databases which would throw an error.

In [3]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-6")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


**Task 1**

From orders.csv, use withColumn() to add three new columns: revenue (unit_price * quantity),

 discounted_price (unit_price * (1 - discount_pct / 100)), and is_delivered (a boolean — true when status equals "Delivered"). 

Show all three new columns alongside order_id.

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
orders = spark.read.csv(
    "s3a://pyspark-30-days-rahul-2026/data/orders.csv",
    header=True,
    inferSchema=True
)

orders.withColumn(
    "revenue",
    col("unit_price") * col("quantity")
).withColumn(
    "discounted_price",
    col("unit_price") * (1 - col("discount_pct") / 100)
).withColumn(
    "is_delivered",
    col("status") == "delivered"
).select(col('order_id'),
col('revenue'),
col('discounted_price'),
col('is_delivered')).show(4)

26/07/28 10:36:37 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------+-------+----------------+------------+
|order_id|revenue|discounted_price|is_delivered|
+--------+-------+----------------+------------+
|   O0001|2599.98|        1169.991|       false|
|   O0002| 449.99|          449.99|       false|
|   O0003|1399.96|        297.4915|       false|
|   O0004| 179.98|         85.4905|       false|
+--------+-------+----------------+------------+
only showing top 4 rows


**Task 2**

Use withColumnRenamed() to rename unit_price to price and customer_id to cust_id.

 Print the column list to verify.

In [5]:
renamed_orders = orders.withColumnRenamed("unit_price", "price") \
    .withColumnRenamed("customer_id", "cust_id")
renamed_orders.columns

['order_id',
 'cust_id',
 'product_id',
 'order_date',
 'quantity',
 'price',
 'discount_pct',
 'status',
 'payment_method',
 'region']

**Task 3**

From customers.csv, drop the columns email, country, and signup_date.

 How many columns remain? Print them.

In [6]:
customers=spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/customers.csv",header=True,inferSchema=True)
dropped_customers=customers.drop("email","country","signup_date")
dropped_customers.columns

['customer_id', 'first_name', 'last_name', 'city', 'state', 'segment']

**Task 4**

Using orders.csv, add a revenue column, then try dropping a column that does not exist — like nonexistent_col.

What happens? Does it throw an error?

In [ ]:
orders.withColumn("revenue",F.col("unit_price") * F.col("quantity")).show(4)
orders.drop('nonexistent_column').show(4) 

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|revenue|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+-------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|2599.98|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West| 449.99|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|          15|Delivered|   Credit Card|Midwest|1399.96|
|   O0004|       C004|      P006|2023-01-12|       2|     89.99|           5|Delivered|    Debit Card|  South| 179.98|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+-------+
only showing top 4 rows
+--------+-----------+--

26/07/29 06:01:31 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 932309 ms exceeds timeout 120000 ms
26/07/29 06:01:32 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/29 06:18:58 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

drop() is designed to be idempotent. If the specified column exists, Spark removes it. If it doesn't exist, Spark simply returns the original DataFrame without throwing an exception. This makes schema modification operations safe and avoids unnecessary failures.

**Rule to remember**

Transformations that only modify the schema (like drop() and withColumnRenamed()) are generally idempotent—if the target column doesn't exist, Spark leaves the DataFrame unchanged.


Transformations that need to read column data (like select(), filter(), withColumn()) must resolve the column, so they throw an AnalysisException if it's missing.